In [43]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as scp
import datetime as dt

In [4]:
hpt_csv = "/Users/tmelman/Documents/Projects/Serif Health DS interview/data/hpt_extract_20250213.csv"
tic_csv = "/Users/tmelman/Documents/Projects/Serif Health DS interview/data/tic_extract_20250213.csv"

In [59]:
hpt = pd.read_csv(hpt_csv)
tic = pd.read_csv(tic_csv)

In [ ]:
# standardize rates
tic['date'] = tic['network_year_month'].apply(lambda x: dt.date(int(str(x)[:4]),int(str(x)[4:]),1))
hpt['date'] = hpt['last_updated_on'].apply(lambda x: dt.date(int(x.split('-')[0]),int(x.split('-')[1]),int(x.split('-')[2])))
# standardize code
hpt['code'] = hpt['raw_code'].apply(lambda x: int(x.split(' ')[-1]))
# standardize payer names
hpt['payer'] = hpt['payer_name'].str.strip().str.lower()
# extract ein from filename
hpt['ein']=hpt['source_file_name'].apply(lambda x: int(x.split('_')[0].replace('-','')[:9]))
hpt = hpt.iloc[6:]

In [69]:
hpt.columns

Index(['source_file_name', 'hospital_id', 'hospital_name', 'last_updated_on',
       'hospital_state', 'license_number', 'payer_name', 'plan_name',
       'code_type', 'raw_code', 'description', 'setting', 'modifiers',
       'standard_charge_gross', 'standard_charge_discounted_cash',
       'standard_charge_negotiated_dollar',
       'standard_charge_negotiated_percentage', 'standard_charge_min',
       'standard_charge_max', 'standard_charge_methodology',
       'additional_payer_notes', 'additional_generic_notes', 'date', 'code',
       'payer', 'ein'],
      dtype='str')

In [70]:
tic.columns

Index(['payer', 'network_name', 'network_id', 'network_year_month',
       'network_region', 'code', 'code_type', 'ein',
       'taxonomy_filtered_npi_list', 'modifier_list', 'billing_class',
       'place_of_service_list', 'negotiation_type', 'arrangement', 'rate',
       'cms_baseline_schedule', 'cms_baseline_rate', 'date'],
      dtype='str')

### Example 1

In [188]:
tic.query('code==43239')

,payer,network_name,network_id,network_year_month,network_region,code,code_type,ein,taxonomy_filtered_npi_list,modifier_list,billing_class,place_of_service_list,negotiation_type,arrangement,rate,cms_baseline_schedule,cms_baseline_rate,date
2,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131740114,"1700348620,1700892056,1922539964,1942685292",NaN,professional,11,negotiated,ffs,993.92,PFS_NONFACILITY_1320202,424.76,2025-01-01
4,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131740114,1346697315,NaN,professional,11,negotiated,ffs,849.63,PFS_NONFACILITY_1320203,391.85,2025-01-01
8,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,"1013948595,1033186093,1033206362,1073545422,12...",NaN,institutional,NaN,negotiated,ffs,6438.00,OPPS,937.56,2025-01-01
10,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,133971298,1801992631,NaN,institutional,NaN,negotiated,ffs,4880.00,OPPS,937.56,2025-01-01
11,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,133971298,"1245635200,1437523537,1528013695,1528432622,15...",NaN,institutional,NaN,negotiated,ffs,5341.00,OPPS,937.56,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,cigna-corporation,national-oap,5dbd8f1c-3f56-4806-917b-e495668bf2bf,202501,USA,43239,CPT,131740114,"1033707500,1043398480,1083445795,1184269912,11...",NaN,professional,"01,02,03,04,05,06,07,08,09,10,11,12,13,14,15,1...",fee schedule,ffs,708.10,PFS_NONFACILITY_1320202,424.76,2025-01-01
214,cigna-corporation,national-oap,5dbd8f1c-3f56-4806-917b-e495668bf2bf,202501,USA,43239,CPT,133971298,"1023470309,1033499181,1033559471,1043283773,10...",NaN,professional,"01,02,03,04,05,06,07,08,09,10,11,12,13,14,15,1...",fee schedule,ffs,796.23,PFS_NONFACILITY_1320202,424.76,2025-01-01
218,cigna-corporation,national-oap,5dbd8f1c-3f56-4806-917b-e495668bf2bf,202501,USA,43239,CPT,131624096,"1124049713,1174551352,1205030228,1578581138",NaN,professional,"02,05,06,07,08,19,21,22,23,24,26,31,34,41,42,5...",fee schedule,ffs,164.00,PFS_FACILITY_1320201,151.42,2025-01-01
219,cigna-corporation,national-oap,5dbd8f1c-3f56-4806-917b-e495668bf2bf,202501,USA,43239,CPT,133971298,"1023320298,1124118831,1134104615,1134325053,11...",NaN,professional,"01,03,04,09,11,12,13,14,15,16,17,18,20,25,32,3...",fee schedule,ffs,772.22,PFS_NONFACILITY_1320202,424.76,2025-01-01


In [203]:
set(tic['payer'].str.lower().str.replace(' ','').str.split('-').apply(lambda x: x[0]))-set(hpt['payer'].str.strip().unique())

{'unitedhealthcare'}

In [212]:
hpt['payer'].str.replace(' ','').unique()

<StringArray>
[                  'aetna',             'healthfirst',
                   'cigna',                   'oscar',
              'healthcare',                  'united',
                  'emblem',                  'humana',
                 'fidelis',             'centerlight',
                'partners',               'northwell',
                 'agewell',               'metroplus',
               'longevity',                     'mvp',
                'hamaspik',                'wellcare',
                  'empire',               'elderplan',
                   'vnsny',                     'nat',
             'centersplan',                     'uhc',
          'villagecaremax',                  'senior',
                  'nippon',                'seiu1199',
                  'molina',               'multiplan',
               'magnacare',                 'centivo',
                  'beacon',        'unitedhealthcare',
            'brighthealth',                 'horizo

In [193]:
set(tic['payer'].unique())-set(hpt['payer'].unique())

{'cigna-corporation', 'unitedhealthcare'}

In [197]:
tic['payer'].unique()

<StringArray>
['unitedhealthcare', 'aetna', 'cigna-corporation']
Length: 3, dtype: str

In [192]:
set(hpt['payer'].unique())-set(tic['payer'].unique())

{'1199',
 'affinity',
 'agewell',
 'american',
 'amida',
 'amida care',
 'amidacare',
 'archcare',
 'bcbs',
 'beacon',
 'bright health',
 'brighton health',
 'centerlight',
 'centersplan',
 'centivo',
 'christian',
 'cigna',
 'elderplan',
 'emblem',
 'empire',
 'empire healthplus',
 'empire medicare advantage',
 'fidelis',
 'firsthealth',
 'ghi',
 'hamaspik',
 'hamaspikchoice',
 'healthcare',
 'healthfirst',
 'healthplus',
 'hip',
 'horizon',
 'humana',
 'independence care',
 'lifetrac',
 'longevity',
 'magnacare',
 'medicaid',
 'medicare',
 'metroplus',
 'molina',
 'multiplan',
 'mvp',
 'nat',
 'nippon',
 'northwell',
 'oscar',
 'oxford',
 'partners',
 'seiu1199',
 'senior',
 'somos',
 'threerivers',
 'uhc',
 'united',
 'united healthcare',
 'villagecaremax',
 'vns',
 'vnsny',
 'wellcare'}

In [187]:
hpt.query('code==43239').iloc[:,6:]

,payer_name,plan_name,code_type,raw_code,description,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,date,code,payer,ein
1,HealthFirst,Commercial Enrollees,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,outpatient,NaN,NaN,NaN,1037.65,NaN,165.40,3206.34,fee schedule,NaN,NaN,2024-07-01,43239,healthfirst,131740114
2,Aetna,Commercial,CPT,43239,UPPER GI ENDOSCOPY BIOPSY,outpatient,NaN,NaN,NaN,1246.73,NaN,1246.73,1394.79,fee schedule,NaN,NaN,2024-07-01,43239,aetna,131740114
4,Oscar,Medicare,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,outpatient,NaN,NaN,NaN,1037.65,NaN,141.77,1815.10,fee schedule,NaN,NaN,2024-07-01,43239,oscar,131740114
6,United,HC NY Essential,CPT,43239,PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE,both,NaN,3925.00,2551.25,161.66,NaN,146.49,1733.49,fee schedule,NaN,NaN,2024-07-01,43239,united,131740114
7,Aetna,Medicare,CPT,43239,PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE,both,NaN,3925.00,2551.25,157.52,NaN,146.49,1733.49,fee schedule,NaN,NaN,2024-07-01,43239,aetna,131740114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2939,magnacare,magnacaredirectplus1372,CPT,43239,HC UGI W BX. SGL/MULTIPLE,both,NaN,1464.39,278.23,9174.00,NaN,NaN,NaN,Case Rate,". MPL 100/75/50% all add ons apply, and lesser...",NaN,2025-01-01,43239,magnacare,133971298
2940,aetna,christianbrothers1640,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,both,NaN,NaN,NaN,8124.66,NaN,NaN,NaN,Case Rate,". Primary procedure pays at 100%, secondary at...",NaN,2025-01-01,43239,aetna,133971298
2941,medicare,aarpmedicarecompletemosaic2009,LOCAL,43239,HEAD HUM 19MM 50MM SHLDR UNIVERS STRL LF CUF A...,both,NaN,32829.50,6237.61,NaN,NaN,NaN,NaN,NaN,Not separately priced,NaN,2025-01-01,43239,medicare,133971298
2945,bcbs,bcbsoutofstatehmo(suitcaselogo)2064,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,both,NaN,NaN,NaN,7334.00,NaN,NaN,NaN,Case Rate,. All Add ons apply except drugs and lesser of...,NaN,2025-01-01,43239,bcbs,133971298


In [71]:
uhc_proc43239_hpt = hpt.query('payer_name=="United Healthcare"').query('raw_code=="43239"')
uhc_proc43239_tic = tic.query('code==43239 and payer=="unitedhealthcare"')


In [180]:
hpt.groupby(['license_number','hospital_name','hospital_id']).size()

license_number  hospital_name              hospital_id                         
13-1740114      Montefiore Medical Center  62915ae8-8d64-4e2f-b05f-b18edde57a3d     380
330024          The Mount Sinai Hospital   5954cbad-a7c5-43f7-b356-8f2ecdad579a     248
7002053H        NYU Langone                40e6a8c8-a68c-4d28-b1d5-fa70d6d09636    2322
dtype: int64

In [219]:
hpt[hpt['standard_charge_negotiated_percentage'].notna()]

,source_file_name,hospital_id,hospital_name,last_updated_on,hospital_state,license_number,payer_name,plan_name,code_type,raw_code,...,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,date,code,payer,ein
176,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Emblem,GHI Network Access,CPT,43239,...,80.0,1233.0,1476.0,percent of total billed charges,NaN,NaN,2024-09-16,43239,emblem,131624096
179,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Multiplan,Complementary Network,CPT,99283,...,68.5,840.0,1680.0,percent of total billed charges,NaN,NaN,2024-09-16,99283,multiplan,131624096
181,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Multiplan,Complementary Network,CPT,99283,...,68.5,840.0,1680.0,percent of total billed charges,NaN,NaN,2024-09-16,99283,multiplan,131624096
184,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Magnacare,JIB,CPT,99283,...,74.0,959.0,1148.0,percent of total billed charges,NaN,NaN,2024-09-16,99283,magnacare,131624096
185,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Emblem,GHI Network Access,CPT,99283,...,80.0,959.0,1148.0,percent of total billed charges,NaN,NaN,2024-09-16,99283,emblem,131624096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2905,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Emblem,GHI Network Access,CPT,99283,...,80.0,1438.5,1722.0,percent of total billed charges,NaN,NaN,2024-09-16,99283,emblem,131624096
2909,131624096_mount-sinai-hospital_standardcharges...,5954cbad-a7c5-43f7-b356-8f2ecdad579a,The Mount Sinai Hospital,2024-09-16,NY,330024,Affinity,Essential Plans 1-2,CPT,99283,...,50.0,840.0,1680.0,percent of total billed charges,NaN,NaN,2024-09-16,99283,affinity,131624096
2930,133971298-1801992631_nyu-langone-tisch_standar...,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemselectcarebronzeexchange1609,LOCAL,43239,...,42.0,NaN,NaN,percent of total billed charges,42 percent of billed charges. Add on payment. ...,NaN,2025-01-01,43239,hip,133971298
2932,133971298-1801992631_nyu-langone-tisch_standar...,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,multiplan,principallifeinsco1255,LOCAL,43239,...,78.0,NaN,NaN,percent of total billed charges,78 percent of billed charges. Add on payment,NaN,2025-01-01,43239,multiplan,133971298


In [181]:
uhc_proc43239_hpt.iloc[:,6:]

,payer_name,plan_name,code_type,raw_code,description,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,date,code,payer,ein
1674,United Healthcare,All Payer,CPT,43239,Egd biopsy single/multiple,outpatient,NaN,NaN,NaN,6438.00,NaN,2513.89,6438.00,case rate,NaN,NaN,2024-09-16,43239,united healthcare,131624096
1707,United Healthcare,"Medicare Advantage, Community Plan Medicare Ad...",CPT,43239,HC Egd Transoral Biopsy Single/Multiple,outpatient,NaN,1800.0,1048.28,1048.28,NaN,127.36,1257.93,Fee Schedule,NaN,NaN,2024-09-16,43239,united healthcare,131624096
1737,United Healthcare,"Medicare Advantage, Community Plan Medicare Ad...",CPT,43239,Egd Transoral Biopsy Single/Multiple - 43239,outpatient,NaN,2200.0,1048.28,1048.28,NaN,153.45,1257.93,Fee Schedule,NaN,NaN,2024-09-16,43239,united healthcare,131624096


In [160]:
uhc_proc43239_tic.query('ein==131624096')

,payer,network_name,network_id,network_year_month,network_region,code,code_type,ein,taxonomy_filtered_npi_list,modifier_list,billing_class,place_of_service_list,negotiation_type,arrangement,rate,cms_baseline_schedule,cms_baseline_rate,date
8,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,"1013948595,1033186093,1033206362,1073545422,12...",NaN,institutional,NaN,negotiated,ffs,6438.00,OPPS,937.56,2025-01-01
12,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,1669440327,NaN,institutional,NaN,negotiated,ffs,2670.00,OPPS,937.56,2025-01-01
14,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,1598095267,NaN,professional,11,negotiated,ffs,862.79,PFS_NONFACILITY_1320201,413.17,2025-01-01
17,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,1710151550,NaN,professional,11,negotiated,ffs,270.93,PFS_NONFACILITY_0610216,373.42,2025-01-01
22,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,"1275531931,1588208201,1700223039,1710324041",NaN,professional,11,negotiated,ffs,514.22,PFS_NONFACILITY_1320201,413.17,2025-01-01
117,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,1710151550,NaN,professional,11,negotiated,ffs,311.55,PFS_NONFACILITY_0610216,373.42,2025-01-01
121,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,"1013948595,1245248939,1316982424",NaN,institutional,NaN,negotiated,ffs,4608.00,OPPS,937.56,2025-01-01
135,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131624096,NaN,NaN,institutional,NaN,negotiated,ffs,1532.00,OPPS,937.56,2025-01-01


### Example 2

In [94]:
hpt.head()

,source_file_name,hospital_id,hospital_name,last_updated_on,hospital_state,license_number,payer_name,plan_name,code_type,raw_code,...,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,date,code,payer,ein
0,13-1740114_montefiore-medical-center_standardc...,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Aetna,Medicare,CPT,99283,...,NaN,83.78,1009.22,fee schedule,NaN,NaN,2024-07-01,99283,aetna,131740114
1,13-1740114_montefiore-medical-center_standardc...,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,HealthFirst,Commercial Enrollees,CPT,43239,...,NaN,165.40,3206.34,fee schedule,NaN,NaN,2024-07-01,43239,healthfirst,131740114
2,13-1740114_montefiore-medical-center_standardc...,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Aetna,Commercial,CPT,43239,...,NaN,1246.73,1394.79,fee schedule,NaN,NaN,2024-07-01,43239,aetna,131740114
3,13-1740114_montefiore-medical-center_standardc...,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Cigna,LocalPlus,CPT,99283,...,NaN,225.00,1797.00,other,NaN,per visit,2024-07-01,99283,cigna,131740114
4,13-1740114_montefiore-medical-center_standardc...,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Oscar,Medicare,CPT,43239,...,NaN,141.77,1815.10,fee schedule,NaN,NaN,2024-07-01,43239,oscar,131740114


In [105]:
hpt.columns

Index(['source_file_name', 'hospital_id', 'hospital_name', 'last_updated_on',
       'hospital_state', 'license_number', 'payer_name', 'plan_name',
       'code_type', 'raw_code', 'description', 'setting', 'modifiers',
       'standard_charge_gross', 'standard_charge_discounted_cash',
       'standard_charge_negotiated_dollar',
       'standard_charge_negotiated_percentage', 'standard_charge_min',
       'standard_charge_max', 'standard_charge_methodology',
       'additional_payer_notes', 'additional_generic_notes', 'date', 'code',
       'payer', 'ein'],
      dtype='str')

In [125]:
tic.columns

Index(['payer', 'network_name', 'network_id', 'network_year_month',
       'network_region', 'code', 'code_type', 'ein',
       'taxonomy_filtered_npi_list', 'modifier_list', 'billing_class',
       'place_of_service_list', 'negotiation_type', 'arrangement', 'rate',
       'cms_baseline_schedule', 'cms_baseline_rate', 'date'],
      dtype='str')

In [97]:
hpt.query('payer=="aetna" and hospital_name=="Montefiore Medical Center" and code_type=="MS-DRG"').iloc[:,-15:]

,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,date,code,payer,ein
32,inpatient,NaN,NaN,NaN,51350.81,NaN,5699.40,51350.81,fee schedule,NaN,NaN,2024-07-01,872,aetna,131740114
1474,inpatient,NaN,NaN,NaN,13434.30,NaN,5699.40,51350.81,fee schedule,NaN,NaN,2024-07-01,872,aetna,131740114
1560,inpatient,NaN,NaN,NaN,45907.79,NaN,24034.78,79673.06,case rate,NaN,NaN,2024-07-01,872,aetna,131740114


In [112]:
for i,row in hpt.iterrows(): 
    if np.any(row==29259.18): print(row); break

source_file_name                         131624096_mount-sinai-hospital_standardcharges...
hospital_id                                           5954cbad-a7c5-43f7-b356-8f2ecdad579a
hospital_name                                                     The Mount Sinai Hospital
last_updated_on                                                                 2024-09-16
hospital_state                                                                          NY
license_number                                                                      330024
payer_name                                                                           Aetna
plan_name                                                                      HMO/POS/PPO
code_type                                                                           MS-DRG
raw_code                                                                               872
description                              SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...

In [130]:
for i,row in tic.iterrows(): 
    if np.any(np.array(["hmo" in str(x) for x in row])): print(row); break

In [121]:
hpt.iloc[217]['description']

'SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITHOUT MCC'

In [182]:
hpt.query('payer=="aetna" and hospital_name=="The Mount Sinai Hospital" and code_type=="MS-DRG"').iloc[:,6:]

,payer_name,plan_name,code_type,raw_code,description,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,date,code,payer,ein
217,Aetna,HMO/POS/PPO,MS-DRG,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,inpatient,NaN,NaN,13241.87,29259.18,NaN,6374.31,38357.42,case rate,NaN,NaN,2024-09-16,872,aetna,131624096
218,Aetna,Medicare,MS-DRG,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,inpatient,NaN,NaN,13241.87,13708.20,NaN,6374.31,38357.42,case rate,NaN,NaN,2024-09-16,872,aetna,131624096
1736,Aetna,"Blackstone, Student Health, Savings Plus",MS-DRG,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,inpatient,NaN,NaN,13241.87,24577.49,NaN,6374.31,38357.42,case rate,NaN,NaN,2024-09-16,872,aetna,131624096
1741,Aetna,Signature Administrators,MS-DRG,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,inpatient,NaN,NaN,13241.87,36573.98,NaN,6374.31,38357.42,case rate,NaN,NaN,2024-09-16,872,aetna,131624096


In [183]:
tic.query('payer=="aetna" and code_type=="MS-DRG" and ein==131624096').sort_values(by='taxonomy_filtered_npi_list')

,payer,network_name,network_id,network_year_month,network_region,code,code_type,ein,taxonomy_filtered_npi_list,modifier_list,billing_class,place_of_service_list,negotiation_type,arrangement,rate,cms_baseline_schedule,cms_baseline_rate,date
30,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1013948595,1245248939,1669476156",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,8757.0,IPPS,6829.75,2025-01-01
179,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1013948595,1245248939,1669476156",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,9206.0,IPPS,6829.75,2025-01-01
145,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1033186093,1033206362,1053721563,1417925181,16...",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,13365.0,IPPS,6829.75,2025-01-01
159,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1033186093,1033206362,1053721563,1417925181,16...",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,17966.0,IPPS,6829.75,2025-01-01
27,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1033206362,1669476156",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,17966.0,IPPS,6829.75,2025-01-01
33,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1033206362,1669476156",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,13365.0,IPPS,6829.75,2025-01-01


In [184]:
tic.loc[[27,33]]

,payer,network_name,network_id,network_year_month,network_region,code,code_type,ein,taxonomy_filtered_npi_list,modifier_list,billing_class,place_of_service_list,negotiation_type,arrangement,rate,cms_baseline_schedule,cms_baseline_rate,date
27,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1033206362,1669476156",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,17966.0,IPPS,6829.75,2025-01-01
33,aetna,open-access-managed-choice,39f0d406-b5df-4046-9759-f08565e45db7,202501,USA,872,MS-DRG,131624096,"1033206362,1669476156",NaN,institutional,"21,31,32,33,34,51,54,55,56,61",negotiated,ffs,13365.0,IPPS,6829.75,2025-01-01
